## Bước 1 — Cài thư viện, tải code và ảnh test

In [ ]:
%uv pip install -q pycocoevalcap pycocotools opencv-python-headless scikit-image scipy pandas matplotlib huggingface_hub install-jdk

import os, sys, json, shutil, subprocess
VOL   = "/mnt/m3diff"
REPO  = "/root/M3Diff"
IMG   = f"{REPO}/datasets/OmniDiff"
OUT   = f"{VOL}/analysis"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

PRED = {
    "Real":   f"{VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl",
    "Render": f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl",
}
for k, p in PRED.items():
    print(("OK    " if os.path.exists(p) else "THIẾU ") + p)

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}

from huggingface_hub import snapshot_download
snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir=IMG,
                  allow_patterns=["Real/test/*", "Render/test/*", "annotation.jsonl"], token=HF_TOKEN or None)

for split in ["Real", "Render"]:
    q = json.load(open(f"{REPO}/data/evaluation/prompt/OmniDiff_{split}_test.json"))
    need = {p for x in q for p in x["image"]}
    miss = [p for p in need if not os.path.exists(f"{IMG}/{p}")]
    print(f"{split}: ảnh cần {len(need)} | thiếu {len(miss)}")

# Xem annotation.jsonl có nhãn loại thay đổi không (để phân tích theo 12 loại ở bước sau)
ann = f"{IMG}/annotation.jsonl"
if os.path.exists(ann):
    with open(ann) as f:
        first = json.loads(f.readline())
    print("\nannotation.jsonl — các trường:", list(first.keys()))
    print(json.dumps(first, ensure_ascii=False)[:600])

## Bước 2 — Java (bộ chấm điểm COCO cần Java)

In [ ]:
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless",
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("java") is None:
    import jdk
    java_home = jdk.install("11", jre=True, path="/root/.jdk")
    os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]
!java -version

## Bước 3 — Script phân tích
Tính cho mỗi cặp ảnh:
- **Kích thước vùng thay đổi:** bản đồ khác biệt màu (ΔE trong không gian Lab, sau khi khớp histogram để loại thay đổi ánh sáng toàn cảnh) → ngưỡng Otsu → mở/đóng hình thái học → vùng liên thông lớn nhất, quy ra **số patch SigLIP** (ô 14×14 px trên ảnh 384×384 model nhìn thấy).
- **Chênh sáng:** hiệu độ sáng trung bình giữa 2 ảnh.
- **Lệch máy:** ghép điểm đặc trưng ORB + RANSAC, ra độ dịch (% cạnh ảnh) và góc xoay.
- **SSIM** giữa 2 ảnh.

Rồi đối chiếu với điểm CIDEr / METEOR / ROUGE-L **từng câu**.

In [ ]:
%%writefile /root/analyze_errors.py
"""Phân tích lỗi M3Diff: điểm từng câu + đặc trưng xử lý ảnh của từng cặp ảnh.

Cho mỗi cặp ảnh (A = ảnh trước, B = ảnh sau) tính:
  - độ lệch máy (ORB + RANSAC), độ chênh sáng, SSIM
  - bản đồ khác biệt (trừ ảnh sau khi cân sáng) -> diện tích vùng thay đổi,
    quy đổi ra số patch SigLIP (14x14 px trên ảnh vào model 384x384)
rồi đối chiếu với điểm CIDEr / METEOR / ROUGE-L của từng câu model sinh ra.
"""
import argparse, json, math, os, sys

import cv2
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from skimage.exposure import match_histograms
from skimage.metrics import structural_similarity

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- màu (bảng màu tham chiếu, 1 series) ----
SURFACE, INK, INK2, MUTED, GRID, BLUE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e6e5e1", "#2a78d6"
WORK_SIDE = 768        # cạnh dài khi xử lý (đủ chi tiết, chạy nhanh)
MODEL_SIDE = 384       # model đệm vuông rồi resize về 384
PATCH = 14             # SigLIP patch 14x14


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--repo", required=True)
    p.add_argument("--questions", required=True)
    p.add_argument("--reference", required=True)
    p.add_argument("--predictions", required=True)
    p.add_argument("--image-root", required=True)
    p.add_argument("--out-dir", required=True)
    return p.parse_args()


# ------------------------------------------------------------------ điểm từng câu
def per_sample_scores(repo, reference, predictions):
    sys.path.insert(0, repo)
    from pathlib import Path
    from metrics import load_predictions
    from pycocotools.coco import COCO
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge

    coco = COCO(reference)
    res = coco.loadRes(load_predictions(Path(predictions)))
    ids = res.getImgIds()
    tok = PTBTokenizer()
    gts = tok.tokenize({i: coco.imgToAnns[i] for i in ids})
    rs = tok.tokenize({i: res.imgToAnns[i] for i in ids})
    keys = list(gts.keys())
    out = pd.DataFrame({"id": keys})
    for scorer, col in [(Cider(), "cider"), (Meteor(), "meteor"), (Rouge(), "rouge_l")]:
        _, scores = scorer.compute_score(gts, rs)
        out[col] = np.asarray(scores, dtype=float)
    out["pred_words"] = [len(res.imgToAnns[i][0]["caption"].split()) for i in keys]
    out["ref_words"] = [len(coco.imgToAnns[i][0]["caption"].split()) for i in keys]
    out["pred"] = [res.imgToAnns[i][0]["caption"] for i in keys]
    out["ref"] = [coco.imgToAnns[i][0]["caption"] for i in keys]
    return out


# ------------------------------------------------------------------ xử lý ảnh
def load_pair(path_a, path_b):
    a = cv2.imread(path_a, cv2.IMREAD_COLOR)
    b = cv2.imread(path_b, cv2.IMREAD_COLOR)
    if a is None or b is None:
        raise FileNotFoundError(path_a if a is None else path_b)
    orig_hw = a.shape[:2]
    if b.shape[:2] != a.shape[:2]:
        b = cv2.resize(b, (a.shape[1], a.shape[0]), interpolation=cv2.INTER_AREA)
    s = WORK_SIDE / max(a.shape[:2])
    if s < 1:
        a = cv2.resize(a, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        b = cv2.resize(b, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return a, b, orig_hw


def camera_shift(ga, gb):
    """Ước lượng lệch máy giữa 2 ảnh: ORB + RANSAC (affine giữ tỉ lệ)."""
    orb = cv2.ORB_create(3000)
    ka, da = orb.detectAndCompute(ga, None)
    kb, db = orb.detectAndCompute(gb, None)
    if da is None or db is None or len(ka) < 10 or len(kb) < 10:
        return np.nan, np.nan, np.nan
    matches = sorted(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(db, da), key=lambda m: m.distance)[:800]
    if len(matches) < 10:
        return np.nan, np.nan, np.nan
    pb = np.float32([kb[m.queryIdx].pt for m in matches])
    pa = np.float32([ka[m.trainIdx].pt for m in matches])
    M, inl = cv2.estimateAffinePartial2D(pb, pa, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    if M is None:
        return np.nan, np.nan, np.nan
    shift_pct = 100 * math.hypot(M[0, 2], M[1, 2]) / max(ga.shape)
    rot_deg = math.degrees(math.atan2(M[1, 0], M[0, 0]))
    return shift_pct, rot_deg, float(inl.mean())


def change_map(a_bgr, b_bgr):
    """Bản đồ thay đổi theo màu (khoảng cách ΔE trong không gian Lab).

    Trước khi trừ, histogram từng kênh Lab của ảnh B được khớp theo ảnh A, để thay đổi ánh sáng toàn cảnh
    không bị tính là thay đổi vật (thay đổi sáng được đo riêng bằng brightness_diff).
    Dùng màu thay vì ảnh xám vì 2 màu khác nhau có thể cùng độ xám.
    """
    la = cv2.cvtColor(a_bgr.astype(np.float32) / 255, cv2.COLOR_BGR2LAB)
    lb = cv2.cvtColor(b_bgr.astype(np.float32) / 255, cv2.COLOR_BGR2LAB)
    lb = match_histograms(lb, la, channel_axis=-1).astype(np.float32)   # cân phân bố màu/sáng của B theo A
    la = cv2.GaussianBlur(la, (5, 5), 0)
    lb = cv2.GaussianBlur(lb, (5, 5), 0)
    delta_e = np.sqrt(((la - lb) ** 2).sum(axis=-1))
    diff = np.clip(delta_e * 2.55, 0, 255).astype(np.uint8)       # ΔE 0..100 -> 0..255
    otsu, _ = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask = cv2.threshold(diff, max(otsu, 30), 255, cv2.THRESH_BINARY)   # sàn ~ΔE 12
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return diff, mask


def pair_features(path_a, path_b):
    a, b, orig_hw = load_pair(path_a, path_b)
    ga = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    shift_pct, rot_deg, inlier = camera_shift(ga, gb)
    _, mask = change_map(a, b)
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    areas = stats[1:, cv2.CC_STAT_AREA] if n > 1 else np.array([0])
    total = mask.size
    # quy đổi diện tích vùng lớn nhất sang số patch SigLIP trên ảnh 384x384 model nhìn thấy
    scale_to_model = MODEL_SIDE / max(mask.shape)
    largest_patches = areas.max() * scale_to_model ** 2 / PATCH ** 2
    return {
        "orig_w": orig_hw[1], "orig_h": orig_hw[0],
        "brightness_diff": float(gb.mean() - ga.mean()),
        "ssim": float(structural_similarity(ga, gb, data_range=255)),
        "change_ratio": float((mask > 0).sum() / total),
        "largest_region_ratio": float(areas.max() / total),
        "largest_region_patches": float(largest_patches),
        "n_regions": int((areas > 0.001 * total).sum()),
        "shift_pct": shift_pct, "rotation_deg": rot_deg, "align_inlier": inlier,
    }


# ------------------------------------------------------------------ biểu đồ
def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(MUTED)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def fmt(v):
    return f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}"


def bar_by_quartile(ax, df, col, title):
    d = df.dropna(subset=[col]).copy()
    d["bin"] = pd.qcut(d[col].rank(method="first"), 4, labels=False)
    g = d.groupby("bin")["cider"].agg(["mean", "count"])
    rng = d.groupby("bin")[col].agg(["min", "max"])
    labels = [f"Nhóm {i+1}\n{fmt(rng.loc[i, 'min'])}–{fmt(rng.loc[i, 'max'])}" for i in g.index]
    x = np.arange(len(g))
    ax.bar(x, g["mean"] * 100, width=0.6, color=BLUE, edgecolor=SURFACE, linewidth=2)
    for xi, (m, c) in zip(x, zip(g["mean"], g["count"])):
        ax.text(xi, m * 100, f"{m*100:.1f}\n(n={c})", ha="center", va="bottom", fontsize=8, color=INK2)
    ax.set_xticks(x, labels, fontsize=8)
    ax.set_title(title, fontsize=10, color=INK, loc="left")
    ax.set_ylabel("CIDEr trung bình", fontsize=9, color=INK2)
    ax.set_ylim(0, max(g["mean"] * 100) * 1.3 + 1)
    style(ax)


def summary_figure(df, name, path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), facecolor=SURFACE)
    bar_by_quartile(axes[0], df, "largest_region_patches", "Kích thước vùng thay đổi lớn nhất\n(số patch SigLIP model nhìn thấy)")
    df = df.assign(abs_brightness=df["brightness_diff"].abs())
    bar_by_quartile(axes[1], df, "abs_brightness", "Chênh độ sáng giữa 2 ảnh\n(|Δ| mức xám 0–255)")
    bar_by_quartile(axes[2], df, "shift_pct", "Độ lệch máy giữa 2 ảnh\n(% cạnh ảnh)")
    fig.suptitle(f"{name}: CIDEr trung bình theo nhóm (chia 4 nhóm đều số cặp, nhóm 1 = nhỏ nhất)",
                 x=0.01, ha="left", fontsize=12, color=INK)
    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor=SURFACE)
    plt.close(fig)


def example_figure(row, img_root, path):
    a, b, _ = load_pair(os.path.join(img_root, row["image_a"]), os.path.join(img_root, row["image_b"]))
    _, mask = change_map(a, b)
    overlay = cv2.cvtColor(b, cv2.COLOR_BGR2RGB).copy()
    overlay[mask > 0] = (0.5 * overlay[mask > 0] + 0.5 * np.array([235, 104, 52])).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), facecolor=SURFACE)
    for ax, im, t in zip(axes, [cv2.cvtColor(a, cv2.COLOR_BGR2RGB), cv2.cvtColor(b, cv2.COLOR_BGR2RGB), overlay],
                         ["Ảnh trước", "Ảnh sau", "Vùng thay đổi (tô cam)"]):
        ax.imshow(im); ax.set_title(t, fontsize=10, color=INK, loc="left"); ax.axis("off")
    short = lambda s, n=220: s if len(s) <= n else s[:n] + "…"
    fig.text(0.01, 0.02, f"Model: {short(row['pred'])}\nNgười viết: {short(row['ref'])}\n"
             f"CIDEr {row['cider']*100:.1f} | vùng lớn nhất ≈ {row['largest_region_patches']:.1f} patch | "
             f"lệch máy {row['shift_pct']:.2f}% | chênh sáng {row['brightness_diff']:+.0f}",
             fontsize=8, color=INK2, va="bottom", wrap=True)
    fig.suptitle(row["id"], x=0.01, ha="left", fontsize=11, color=INK)
    fig.tight_layout(rect=(0, 0.14, 1, 1))
    fig.savefig(path, dpi=130, facecolor=SURFACE)
    plt.close(fig)


# ------------------------------------------------------------------ main
def main():
    args = parse()
    os.makedirs(args.out_dir, exist_ok=True)
    questions = {q["id"]: q["image"] for q in json.load(open(args.questions))}

    print(f"[{args.name}] chấm điểm từng câu...", flush=True)
    df = per_sample_scores(args.repo, args.reference, args.predictions)

    print(f"[{args.name}] tính đặc trưng ảnh cho {len(df)} cặp...", flush=True)
    feats = []
    for k, qid in enumerate(df["id"]):
        img_a, img_b = questions[qid]
        f = pair_features(os.path.join(args.image_root, img_a), os.path.join(args.image_root, img_b))
        f.update(id=qid, image_a=img_a, image_b=img_b)
        feats.append(f)
        if (k + 1) % 100 == 0:
            print(f"  {k+1}/{len(df)}", flush=True)
    df = df.merge(pd.DataFrame(feats), on="id")
    df["degenerate"] = df["pred_words"] > 300
    df.to_csv(os.path.join(args.out_dir, "per_sample.csv"), index=False)

    # tương quan hạng Spearman giữa CIDEr và từng đặc trưng
    rows = []
    for col, label in [("largest_region_patches", "Kích thước vùng thay đổi (patch)"),
                       ("change_ratio", "Tỉ lệ diện tích thay đổi"),
                       ("n_regions", "Số vùng thay đổi"),
                       ("ssim", "SSIM (giống nhau)"),
                       ("brightness_diff", "Chênh sáng (có dấu)"),
                       ("shift_pct", "Lệch máy (%)"),
                       ("pred_words", "Độ dài câu model")]:
        d = df[["cider", col]].dropna()
        rho, p = spearmanr(d["cider"], d[col])
        rows.append({"đặc trưng": label, "rho": round(float(rho), 3), "p": float(p), "n": len(d)})
    corr = pd.DataFrame(rows)
    corr.to_csv(os.path.join(args.out_dir, "correlations.csv"), index=False)

    summary_figure(df, args.name, os.path.join(args.out_dir, "summary.png"))
    worst = df[~df["degenerate"]].nsmallest(3, "cider")
    for _, r in worst.iterrows():
        example_figure(r, args.image_root, os.path.join(args.out_dir, f"worst_{r['id']}.png"))

    print(f"\n== {args.name} ==")
    print(f"Số cặp: {len(df)} | CIDEr trung bình từng câu: {df['cider'].mean()*100:.1f} | câu bị lặp (>300 từ): {int(df['degenerate'].sum())}")
    print(f"Lệch máy: trung vị {df['shift_pct'].median():.2f}% cạnh ảnh, {int((df['shift_pct'] > 1).sum())} cặp lệch > 1%")
    print(f"Vùng thay đổi lớn nhất: trung vị {df['largest_region_patches'].median():.1f} patch SigLIP "
          f"({int((df['largest_region_patches'] < 4).sum())} cặp < 4 patch)")
    print("\nTương quan Spearman với CIDEr từng câu (rho > 0: đặc trưng càng lớn điểm càng cao):")
    print(corr.to_string(index=False))
    print(f"\nĐã lưu vào {args.out_dir}: per_sample.csv, correlations.csv, summary.png, worst_*.png")


if __name__ == "__main__":
    main()

## Bước 4 — Chạy phân tích Real và Render (vài phút mỗi bộ)

In [ ]:
from IPython.display import Image, display
import glob

for split in ["Real", "Render"]:
    out, pred = f"{OUT}/{split}", PRED[split]
    !{sys.executable} /root/analyze_errors.py --name OmniDiff-{split} --repo {REPO} \
      --questions {REPO}/data/evaluation/prompt/OmniDiff_{split}_test.json \
      --reference {REPO}/data/evaluation/reference/OmniDiff_{split}.json \
      --predictions {pred} --image-root {IMG} --out-dir {out} 2>&1 | grep -v "tokenized\|index\|Loading\|DONE\|Done"
    display(Image(f"{out}/summary.png"))
    for p in sorted(glob.glob(f"{out}/worst_*.png")):
        display(Image(p))